# 03 · Validación del Elo y de la probabilidad pre-partido

Cada partido se pronostica con el Elo de antes de jugarse (sin datos del futuro) y se compara contra un pronóstico ingenuo (las frecuencias de local, empate y visitante) con Brier y log-loss. Al final, la curva de calibración.

In [1]:
import sys, statistics, math
sys.path.insert(0, '.')
from datos import api, league_games, result, log_loss, elo_probabilities, m, RESULTS
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [2]:
juegos = league_games()
QUEMA = 60
empates = sum(result(g) == 'E' for g in juegos[:QUEMA]) / QUEMA
probs = elo_probabilities(juegos, m.ELO_K, m.ELO_HOME_ADVANTAGE, empates)[QUEMA:]
evaluados = juegos[QUEMA:]
frec = tuple(sum(result(g) == r for g in juegos[:QUEMA]) / QUEMA for r in RESULTS)
for nombre, ps in (('Elo', probs), ('Frecuencias', [frec] * len(evaluados))):
    b = statistics.mean(m.brier(p, result(g)) for p, g in zip(ps, evaluados))
    ll = statistics.mean(log_loss(p, result(g)) for p, g in zip(ps, evaluados))
    print(f"{nombre:12s} Brier {b:.4f} · log-loss {ll:.4f}")

Elo          Brier 0.6472 · log-loss 1.0718
Frecuencias  Brier 0.6606 · log-loss 1.0908


In [3]:
cajas = [[] for _ in range(5)]
for p, g in zip(probs, evaluados):
    cajas[min(int(p[0] * 5), 4)].append((p[0], result(g) == 'G'))
xs, ys = [], []
for c in cajas:
    if c:
        xs.append(statistics.mean(a for a, _ in c)); ys.append(sum(b for _, b in c) / len(c))
        print(f"Pronóstico medio {xs[-1]:.2f} · frecuencia real {ys[-1]:.2f} · {len(c)} partidos")
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot([0, 1], [0, 1], color='#9AA09A', linestyle='--')
ax.plot(xs, ys, marker='o', color='#328336')
ax.set_xlabel('Probabilidad de victoria local (Elo)'); ax.set_ylabel('Frecuencia observada'); ax.set_title('Curva de calibración')
plt.show()

Pronóstico medio 0.37 · frecuencia real 0.20 · 44 partidos
Pronóstico medio 0.50 · frecuencia real 0.42 · 321 partidos
Pronóstico medio 0.65 · frecuencia real 0.67 · 39 partidos


C:\Users\Pato\AppData\Local\Temp\ipykernel_32256\3065929266.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura.** El Elo agrega valor si su Brier y su log-loss quedan por debajo de los del pronóstico ingenuo. Una curva por encima de la diagonal indica que el Elo subestima al local; por debajo, que lo sobreestima. Cada punto lleva su n: con una temporada son pocos partidos.